# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² Mass Spectrometry Analysis dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and their fields with @id references
record_sets = metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    print(f"  Description: {rs.get('description', 'N/A')}")
    if 'field' in rs:
        print("  Fields:")
        for fld in rs['field']:
            if isinstance(fld, dict):
                print(f"    - ID: {fld['@id']}, Name: {fld.get('name', 'N/A')}, DataType: {fld.get('dataType', 'N/A')}")
            else:
                print(f"    - Field reference: {fld}")
    print("")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For demonstration, we will extract the main record set.
# You should use the specific `@id` from the overview. If no explicit record sets, try typical values or list them above.

# Example: Get all record set IDs
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"\nExtracting records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        if not df.empty:
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} rows, columns: {df.columns.tolist()}")
        else:
            print(f"No records in dataframe for {record_set_id}.")
    else:
        print("No records found in this record set.")

# For further code, we'll try to use the first non-empty dataframe
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    print(f"\nFields (@id) for main record set: {main_record_set_id}")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    main_record_set_id = None
    print("No tabular record sets could be loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping data by key attributes.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings("ignore")

# Pick a numeric field for analysis by searching for numeric columns
if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    # Try to select a field likely to be numeric: 'MW' (Molecular Weight) or 'coverage_percent' or similar
    # List columns
    print("Available columns in main record set:")
    print(df.columns.tolist())
    # Heuristically pick first float/int column
    numeric_field = None
    for col in df.columns:
        # Try to cast to numeric and see if it's not null
        try:
            vals = pd.to_numeric(df[col], errors='coerce')
            if (vals.notnull().sum() > 0):
                numeric_field = col
                print(f"Using numeric field for demo: {numeric_field}")
                df[numeric_field] = vals
                break
        except Exception:
            continue
    if numeric_field is None:
        print("No numeric field found for analysis.")
    else:
        threshold = df[numeric_field].mean() if np.isfinite(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} rows")
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() else 1)
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try to group by a categorical field (e.g. 'description' or 'accession')
        candidate_groups = [c for c in df.columns if c not in [numeric_field] and df[c].dtype == object]
        group_field = candidate_groups[0] if candidate_groups else None
        if group_field:
            grouped = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().head()
            print(f"Mean {numeric_field} grouped by {group_field} (top 5):")
            print(grouped)
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and numeric_field is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(dataframes[main_record_set_id][numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    # Boxplot by group field, if available
    if group_field is not None:
        plt.figure(figsize=(10, 5))
        top_groups = dataframes[main_record_set_id][group_field].value_counts().index[:5]
        sns.boxplot(x=group_field, y=numeric_field, data=dataframes[main_record_set_id][dataframes[main_record_set_id][group_field].isin(top_groups)])
        plt.title(f"Boxplot of {numeric_field} by {group_field} (top 5)")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Using Croissant's standardized metadata, we explored available record sets, their fields, and extracted protein data using mlcroissant directly from FAIR² schema.
- We identified core numeric fields (such as molecular weight or coverage percent), performed basic normalization, filtering, and visualized distributions.
- The dataset provides useful protein-level attributes suitable for various bioinformatics analyses, and its schema makes downstream integration easy and reproducible.

Continue with domain-specific tasks such as sequence analysis, modification profiling, or direct hypothesis testing using these clean dataframes!